# Конфигурация экспериментов

UCI HAR, 9 каналов, 128 шагов, 6 классов. Общая длительность -- 2.56 с.
Частоты 10--50 Гц. В каждом train-канале окна один отказ 0.64 с.
Test -- те же разные частоты и отказы 0 / 0.64 / 1.28 с.

3-fold GroupKFold по людям. Нормализация и mean/median -- внутри fit fold.
Выбор эпох -- по CV macro F1, затем обучение с нуля на всем official train.
Test изолирован от выбора параметров. Seed 42; fold seeds 43/44/45.
AdamW, lr 8e-4, weight decay 1e-3, batch 128, 35 эпох, patience 7.
Конкретное устройство и версии пакетов сохраняются каждым запуском в `results/*_config.json`.

Четыре заполнения одного Pairwise EdgeMTSC и четыре варианта CTF-inspired adapter
перед тем же backbone. Все восемь конфигураций обучаются независимо.
Абляции сохраняют число параметров adapter: no patch dropout, no token exchange,
filled tokens. Сравнение CNN и adapter не является абляцией при одинаковом числе параметров.

Метрики: accuracy, balanced accuracy, macro F1, параметры, final/CV train time,
forward latency и CPU preparation time. Все определения и ограничения -- в `REPORT.md`.


## Проверка перед передачей

Формат и синтаксис всех ноутбуков проверены. Подготовка и EDA выполнены
на полном UCI HAR из предоставленного архива. Проверены оба варианта упаковки
данных: файлы в zip и вложенный UCI HAR Dataset.zip.

Проверена автоматическая сборка отчета, сохранение checkpoint, таблиц и графиков.
В приложенном состоянии результаты получены полным запуском; при повторном запуске
устройство и версии пакетов заново попадут в `results/*_config.json`.

Отдельно проверены разделение людей, длины и вложенность пропусков,
все четыре заполнения, полностью пустые каналы, конечность logits и градиентов,
неизменность вывода при подмене значений под маской, отсутствие обхода patch dropout
через прямой вход CNN и одинаковое число параметров в четырех вариантах adapter.
Итоговые checkpoints, метрики, таблицы и графики сохранены; временные проверочные артефакты в проект не включены.

| Требование | Где выполнено |
| --- | --- |
| Вычислительная конфигурация | 02 и 03, results/*_config.json |
| Корпус, разные частоты и периоды пропусков | 00 |
| Визуализация частот, пропусков и статистики | 01 |
| Четыре заполнения и отдельные обучения baseline | 02 |
| Раздельное время, channel tokens и маскирование | 03 |
| Качество, параметры, время и графики | 02, 03 |
| Дополнительные test-сценарии и абляции | 03 |
| Выводы по фактическому запуску | Последняя ячейка 03 и REPORT.md |


Интерпретация результата: median-imputation оказалась сильной при умеренных пропусках, но при 50% непрерывного outage CTF-inspired обработка устойчивее. Это связываю с характером коротких инерциальных окон UCI HAR и не обобщаю на произвольные временные ряды.